# Power BI REST API - Python Examples

Interactive notebook demonstrating Power BI REST API operations using Python.

## Prerequisites
```bash
pip install msal requests pandas
```

In [ ]:
import os
import json
import requests
import pandas as pd
from msal import ConfidentialClientApplication
from datetime import datetime

## Configuration

Set your Azure AD and Power BI configuration:

In [ ]:
# Configuration - use environment variables in production
CONFIG = {
    'tenant_id': os.getenv('PBI_TENANT_ID', 'your-tenant-id'),
    'client_id': os.getenv('PBI_APP_ID', 'your-app-id'),
    'client_secret': os.getenv('PBI_CLIENT_SECRET', 'your-secret'),
    'scope': ['https://analysis.windows.net/powerbi/api/.default'],
    'api_url': 'https://api.powerbi.com/v1.0/myorg'
}

print(f"Configured for tenant: {CONFIG['tenant_id'][:8]}...")

## Authentication

In [ ]:
class PowerBIClient:
    """Simple Power BI REST API client."""
    
    def __init__(self, config):
        self.config = config
        self.access_token = None
        self._authenticate()
    
    def _authenticate(self):
        """Authenticate using MSAL."""
        authority = f"https://login.microsoftonline.com/{self.config['tenant_id']}"
        
        app = ConfidentialClientApplication(
            self.config['client_id'],
            authority=authority,
            client_credential=self.config['client_secret']
        )
        
        result = app.acquire_token_for_client(scopes=self.config['scope'])
        
        if 'access_token' in result:
            self.access_token = result['access_token']
            print("✓ Authentication successful")
        else:
            raise Exception(f"Authentication failed: {result.get('error_description', 'Unknown error')}")
    
    def _get_headers(self):
        return {
            'Authorization': f'Bearer {self.access_token}',
            'Content-Type': 'application/json'
        }
    
    def get(self, endpoint):
        """GET request to Power BI API."""
        url = f"{self.config['api_url']}/{endpoint}"
        response = requests.get(url, headers=self._get_headers())
        response.raise_for_status()
        return response.json()
    
    def post(self, endpoint, data=None):
        """POST request to Power BI API."""
        url = f"{self.config['api_url']}/{endpoint}"
        response = requests.post(url, headers=self._get_headers(), json=data)
        response.raise_for_status()
        return response.json() if response.text else None

# Initialize client
pbi = PowerBIClient(CONFIG)

## Workspace Operations

In [ ]:
# List all workspaces
workspaces = pbi.get('groups')
df_workspaces = pd.DataFrame(workspaces['value'])

print(f"Found {len(df_workspaces)} workspaces\n")
df_workspaces[['name', 'id', 'isOnDedicatedCapacity']].head(10)

In [ ]:
# Get workspace details
workspace_id = df_workspaces.iloc[0]['id']  # First workspace
workspace_name = df_workspaces.iloc[0]['name']

print(f"Selected workspace: {workspace_name}")
print(f"Workspace ID: {workspace_id}")

## Dataset Operations

In [ ]:
# List datasets in workspace
datasets = pbi.get(f'groups/{workspace_id}/datasets')
df_datasets = pd.DataFrame(datasets['value'])

print(f"Found {len(df_datasets)} datasets in {workspace_name}\n")
if not df_datasets.empty:
    display(df_datasets[['name', 'id', 'isRefreshable', 'configuredBy']].head())

In [ ]:
# Get refresh history for a dataset
if not df_datasets.empty:
    dataset_id = df_datasets.iloc[0]['id']
    dataset_name = df_datasets.iloc[0]['name']
    
    refreshes = pbi.get(f'groups/{workspace_id}/datasets/{dataset_id}/refreshes?$top=10')
    
    print(f"Refresh history for: {dataset_name}\n")
    if refreshes['value']:
        df_refreshes = pd.DataFrame(refreshes['value'])
        display(df_refreshes[['startTime', 'endTime', 'status']].head())
    else:
        print("No refresh history found")

In [ ]:
# Trigger dataset refresh (uncomment to execute)
# response = pbi.post(f'groups/{workspace_id}/datasets/{dataset_id}/refreshes')
# print("Refresh triggered successfully")

## Report Operations

In [ ]:
# List reports in workspace
reports = pbi.get(f'groups/{workspace_id}/reports')
df_reports = pd.DataFrame(reports['value'])

print(f"Found {len(df_reports)} reports in {workspace_name}\n")
if not df_reports.empty:
    display(df_reports[['name', 'id', 'datasetId', 'webUrl']].head())

## Tenant-Wide Inventory

In [ ]:
def get_tenant_inventory(client):
    """Generate tenant-wide inventory."""
    
    inventory = {
        'scan_time': datetime.now().isoformat(),
        'workspaces': [],
        'datasets': [],
        'reports': []
    }
    
    # Get all workspaces
    workspaces = client.get('groups')['value']
    inventory['workspaces'] = workspaces
    
    # Get datasets and reports per workspace
    for ws in workspaces:
        ws_id = ws['id']
        ws_name = ws['name']
        
        try:
            # Datasets
            datasets = client.get(f'groups/{ws_id}/datasets')['value']
            for ds in datasets:
                ds['workspaceId'] = ws_id
                ds['workspaceName'] = ws_name
            inventory['datasets'].extend(datasets)
            
            # Reports
            reports = client.get(f'groups/{ws_id}/reports')['value']
            for rpt in reports:
                rpt['workspaceId'] = ws_id
                rpt['workspaceName'] = ws_name
            inventory['reports'].extend(reports)
            
        except Exception as e:
            print(f"Could not scan {ws_name}: {e}")
    
    return inventory

# Generate inventory
inventory = get_tenant_inventory(pbi)

print(f"Inventory Summary:")
print(f"  Workspaces: {len(inventory['workspaces'])}")
print(f"  Datasets: {len(inventory['datasets'])}")
print(f"  Reports: {len(inventory['reports'])}")

In [ ]:
# Export inventory to CSV
pd.DataFrame(inventory['workspaces']).to_csv('workspaces.csv', index=False)
pd.DataFrame(inventory['datasets']).to_csv('datasets.csv', index=False)
pd.DataFrame(inventory['reports']).to_csv('reports.csv', index=False)

print("Inventory exported to CSV files")

## Execute DAX Query (Premium/Fabric Only)

In [ ]:
def execute_dax_query(client, workspace_id, dataset_id, dax_query):
    """Execute DAX query using REST API (Premium/Fabric only)."""
    
    endpoint = f'groups/{workspace_id}/datasets/{dataset_id}/executeQueries'
    
    payload = {
        'queries': [{'query': dax_query}],
        'serializerSettings': {'includeNulls': True}
    }
    
    result = client.post(endpoint, payload)
    
    if result and 'results' in result:
        tables = result['results'][0].get('tables', [])
        if tables:
            rows = tables[0].get('rows', [])
            return pd.DataFrame(rows)
    
    return pd.DataFrame()

# Example DAX query (uncomment and modify)
# dax_query = '''
# EVALUATE
# SUMMARIZECOLUMNS(
#     'Date'[Year],
#     "Total Sales", [Total Sales]
# )
# '''
# 
# result_df = execute_dax_query(pbi, workspace_id, dataset_id, dax_query)
# display(result_df)

## Bulk Operations

In [ ]:
def refresh_all_datasets(client, workspace_id, skip_on_error=True):
    """Trigger refresh for all refreshable datasets in a workspace."""
    
    datasets = client.get(f'groups/{workspace_id}/datasets')['value']
    refreshable = [d for d in datasets if d.get('isRefreshable', False)]
    
    results = []
    
    for ds in refreshable:
        try:
            client.post(f"groups/{workspace_id}/datasets/{ds['id']}/refreshes")
            results.append({'name': ds['name'], 'status': 'triggered'})
            print(f"✓ Triggered: {ds['name']}")
        except Exception as e:
            results.append({'name': ds['name'], 'status': 'failed', 'error': str(e)})
            print(f"✗ Failed: {ds['name']} - {e}")
            if not skip_on_error:
                raise
    
    return pd.DataFrame(results)

# Refresh all datasets (uncomment to execute)
# refresh_results = refresh_all_datasets(pbi, workspace_id)
# display(refresh_results)

## Cleanup

In [ ]:
print("Notebook execution complete")
print(f"Timestamp: {datetime.now().isoformat()}")